# OCR comparison for typeplate recognition

Bachelor thesis: *Automated Typeplate Recognition Using OCR and AI* — Mitchel Wouters (15235076), UvA.

This notebook runs the three experiments reported in the thesis:

1. **Experiment 1** — Tesseract baseline vs. EasyOCR vs. PaddleOCR on raw images.
2. **Experiment 2** — Effect of image preprocessing (Hough deskew + CLAHE) on all three models.
3. **Experiment 3** — Offline best model vs. an online vision-language model (optional).

Results are saved to `results/` as CSV and PNG files for use in the thesis.


## 1. Setup and configuration

In [ ]:
# ------------------------------------------------------------------
# Run this cell first every time you open the notebook in Colab.
# It mounts your Google Drive and installs the required packages.
# ------------------------------------------------------------------
import os, sys

# ---- Mount Google Drive ------------------------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ---- Install packages (only needed once per Colab session) -------
os.system('apt-get install -y tesseract-ocr > /dev/null 2>&1')
os.system('pip install pytesseract easyocr paddleocr paddlepaddle '
          'jiwer openpyxl pillow opencv-python-headless -q')

print("Setup complete.")
print("Python:", sys.version.split()[0])


In [ ]:
# Core libraries
import os
import re
import sys
import time
import json
import unicodedata
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

warnings.filterwarnings("ignore")
np.random.seed(42)

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


In [ ]:
# OCR libraries
import pytesseract
print("pytesseract:", pytesseract.__version__)
try:
    print("Tesseract version:", pytesseract.get_tesseract_version())
except Exception as e:
    print("Tesseract not found:", e)

import easyocr
print("easyocr available")

from paddleocr import PaddleOCR
print("paddleocr available")

from jiwer import cer, wer
print("jiwer available")

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x


In [ ]:
# ==================================================================
# PATHS — matches your Google Drive layout:
#
#   My Drive/
#     labeled_testset.xlsx        <-- directly in root
#     testset/                    <-- folder with all 100 images
#     ocr_results/                <-- created automatically for output
# ==================================================================
DRIVE_ROOT = Path("/content/drive/MyDrive")

TESTSET_XLSX       = DRIVE_ROOT / "labeled_testset.xlsx"
TESTSET_IMAGES_DIR = DRIVE_ROOT / "testset"
OUTPUT_DIR         = DRIVE_ROOT / "ocr_results"
CACHE_DIR          = DRIVE_ROOT / "ocr_results" / "cache"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Quick sanity check
print("labeled_testset.xlsx:", TESTSET_XLSX.exists())
print("testset/ folder:", TESTSET_IMAGES_DIR.exists())
if TESTSET_IMAGES_DIR.exists():
    imgs = [f for f in TESTSET_IMAGES_DIR.iterdir()
            if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
    print(f"Images found: {len(imgs)}")
    if imgs:
        print("  First 3:", sorted(f.name for f in imgs)[:3])


## 2. Load and normalise testset

The labelled testset contains 100 images with ground truth values for five fields plus a full transcription. Some columns contain inconsistent label values (`metal_plate` vs `metal plate`, `OK` vs `complete`, etc.) and missing values are stored as the string `"NaN"`. Both are normalised here.

In [ ]:
def is_missing(v):
    """Robust missing check: catches None, numpy NaN and the literal string 'NaN'."""
    if v is None:
        return True
    if isinstance(v, float) and pd.isna(v):
        return True
    if isinstance(v, str) and v.strip().lower() in {"", "nan"}:
        return True
    return False


def to_str_or_none(v):
    """Convert to clean string, or None if missing."""
    if is_missing(v):
        return None
    return str(v).strip()


def normalise_status(s):
    """Collapse the five raw status values to OK / CHECK."""
    if is_missing(s):
        return "CHECK"
    s = str(s).strip().lower()
    if s in {"ok", "complete"}:
        return "OK"
    return "CHECK"


def normalise_plate_type(s):
    """Normalise 'metal plate' -> 'metal_plate'."""
    if is_missing(s):
        return "unknown"
    return str(s).strip().lower().replace(" ", "_")


df = pd.read_excel(TESTSET_XLSX)
df["status"] = df["status"].apply(normalise_status)
df["plate_type"] = df["plate_type"].apply(normalise_plate_type)
df["transcription"] = df["transcription"].apply(lambda x: "" if is_missing(x) else str(x))
# Note: we deliberately leave brand/model/etc as-is in the DataFrame because pandas
# converts None back to NaN. We use to_str_or_none() when consuming individual values.

print(f"Loaded {len(df)} rows.")
print("Status:", df["status"].value_counts().to_dict())
print("Plate type:", df["plate_type"].value_counts().to_dict())
print("Readability:", df["readability"].value_counts().to_dict())

# Sanity: per field missingness
for col in ["brand", "model", "serial_number", "build_year", "power_output"]:
    missing = df[col].apply(is_missing).sum()
    print(f"  {col}: {missing}/{len(df)} missing ({missing/len(df)*100:.0f}%)")


## 3. Dataset statistics

Five figures describe the testset composition. These feed Section 4.1 of the thesis.

In [ ]:
# ----- Figure 1: missing values ---------------------------------
fields = ["brand", "model", "serial_number", "build_year", "power_output", "transcription"]
missing_pct = {f: df[f].apply(is_missing).sum() / len(df) * 100 for f in fields}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
heat = pd.DataFrame({f: df[f].apply(lambda x: 1 if is_missing(x) else 0) for f in fields})
axes[0].imshow(heat.values, aspect="auto", cmap="Greys", interpolation="nearest")
axes[0].set_yticks([])
axes[0].set_xticks(range(len(fields)))
axes[0].set_xticklabels(fields, rotation=30, ha="right")
axes[0].set_title("Per row missingness heatmap")
axes[0].set_ylabel("Rows")

bars = axes[1].barh(list(missing_pct.keys())[::-1], list(missing_pct.values())[::-1], color="#444")
axes[1].set_xlim(0, 100)
axes[1].set_xlabel("Percentage missing")
axes[1].set_title("Missing values per field")
for bar, val in zip(bars, list(missing_pct.values())[::-1]):
    axes[1].text(val + 1, bar.get_y() + bar.get_height()/2, f"{val:.0f}%", va="center")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig01_missing_values.png", dpi=150, bbox_inches="tight")
plt.show()
print({f: f"{v:.0f}%" for f, v in missing_pct.items()})


In [ ]:
# ----- Figure 2: testset composition ----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

status_counts = df["status"].value_counts()
axes[0].pie(status_counts.values, labels=status_counts.index, autopct="%1.0f%%",
            colors=["#5cb85c", "#f0ad4e"])
axes[0].set_title("Status (OK vs CHECK)")

plate_counts = df["plate_type"].value_counts()
axes[1].bar(plate_counts.index, plate_counts.values, color="#337ab7")
axes[1].set_title("Plate type")
axes[1].set_ylabel("Count")
for i, v in enumerate(plate_counts.values):
    axes[1].text(i, v + 0.5, str(v), ha="center")

read_order = ["good", "medium", "poor"]
read_counts = df["readability"].value_counts().reindex(read_order, fill_value=0)
axes[2].bar(read_counts.index, read_counts.values, color=["#5cb85c", "#f0ad4e", "#d9534f"])
axes[2].set_title("Readability")
axes[2].set_ylabel("Count")
for i, v in enumerate(read_counts.values):
    axes[2].text(i, v + 0.5, str(v), ha="center")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig02_testset_composition.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ----- Figure 3: status and readability by plate type -----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ct1 = pd.crosstab(df["plate_type"], df["status"])
ct1.plot(kind="bar", ax=axes[0], color=["#f0ad4e", "#5cb85c"])
axes[0].set_title("Status by plate type")
axes[0].set_xlabel("Plate type")
axes[0].set_ylabel("Count")
axes[0].legend(title="Status")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

ct2 = pd.crosstab(df["plate_type"], df["readability"])
ct2 = ct2.reindex(columns=[c for c in ["good", "medium", "poor"] if c in ct2.columns])
ct2.plot(kind="bar", ax=axes[1], color=["#5cb85c", "#f0ad4e", "#d9534f"])
axes[1].set_title("Readability by plate type")
axes[1].set_xlabel("Plate type")
axes[1].set_ylabel("Count")
axes[1].legend(title="Readability")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig03_crosstabs.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ----- Figure 4: completeness by plate type ---------------------
fields_to_plot = ["brand", "model", "serial_number", "build_year", "power_output"]
plate_types = df["plate_type"].unique()

completeness = {}
for pt in plate_types:
    sub = df[df["plate_type"] == pt]
    completeness[pt] = {f: (sub[f].apply(lambda x: not is_missing(x)).sum() / len(sub) * 100) for f in fields_to_plot}

comp_df = pd.DataFrame(completeness).T

fig, ax = plt.subplots(figsize=(11, 5))
comp_df.plot(kind="bar", ax=ax, colormap="tab10")
ax.set_ylabel("Percentage present")
ax.axhline(100, ls="--", color="grey", alpha=0.5)
ax.set_ylim(0, 110)
ax.set_title("Ground truth completeness per field, by plate type")
ax.legend(loc="lower right", ncol=3)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig04_completeness_by_platetype.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ----- Figure 5: build year and transcription length ------------
years = pd.to_numeric(df["build_year"], errors="coerce").dropna()
lengths = df["transcription"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(years, bins=range(1940, 2030, 5), color="#337ab7", edgecolor="white")
axes[0].set_title(f"Build year distribution (n={len(years)} non missing)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Count")

axes[1].hist(lengths, bins=20, color="#5cb85c", edgecolor="white")
axes[1].axvline(lengths.median(), color="red", linestyle="--",
                label=f"median = {int(lengths.median())} chars")
axes[1].set_title("Transcription length")
axes[1].set_xlabel("Characters per plate")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig05_year_and_length.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Evaluation utilities

Three groups of metrics are computed for every model run:

- **Raw text**: CER and WER between full OCR output and the manual transcription.
- **Field level**: precision, recall and F1 per field after heuristic extraction.
- **Format validity**: whether the extracted value matches the expected pattern for the field, independent of exact match.

All metrics use `jiwer` 4.0.0 for edit distance and standard Unicode normalisation for string comparison.

In [ ]:
def normalise_text(s):
    """Unicode NFKC, lowercase, collapse whitespace."""
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def safe_cer(ref, hyp):
    ref, hyp = normalise_text(ref), normalise_text(hyp)
    if not ref:
        return 1.0 if hyp else 0.0
    try:
        return float(cer(ref, hyp))
    except Exception:
        return 1.0


def safe_wer(ref, hyp):
    ref, hyp = normalise_text(ref), normalise_text(hyp)
    if not ref:
        return 1.0 if hyp else 0.0
    try:
        return float(wer(ref, hyp))
    except Exception:
        return 1.0


def field_compare(extracted, ground_truth):
    """Return True if the two values match after light normalisation."""
    a = normalise_text(extracted)
    b = normalise_text(ground_truth)
    if not a or not b:
        return False
    # Allow trailing punctuation, internal hyphen / space differences
    a = re.sub(r"[^a-z0-9]", "", a)
    b = re.sub(r"[^a-z0-9]", "", b)
    return a == b


def compute_field_metrics(rows, gt_col, pred_col):
    """Compute precision, recall, F1 over a list of dict rows."""
    tp = fp = fn = 0
    for r in rows:
        gt = r.get(gt_col)
        pr = r.get(pred_col)
        gt_present = not is_missing(gt)
        pr_present = not is_missing(pr)
        if gt_present and pr_present:
            if field_compare(pr, gt):
                tp += 1
            else:
                fp += 1
                fn += 1
        elif gt_present and not pr_present:
            fn += 1
        elif not gt_present and pr_present:
            fp += 1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn}


In [ ]:
# ----- Format-valid patterns ------------------------------------
YEAR_RE = re.compile(r"\b(19[5-9]\d|20[0-2]\d)\b")
POWER_RE = re.compile(r"\b\d+(?:[.,]\d+)?\s*(?:kw|kva|hp|w|v|a|hz)\b", re.IGNORECASE)
SERIAL_RE = re.compile(r"\b[A-Z0-9][A-Z0-9\-]{4,}\b")
MODEL_RE = re.compile(r"\b[A-Z0-9][A-Z0-9\-/]{2,}\b")

def format_valid(field, value):
    if not value:
        return False
    s = str(value).strip()
    if field == "build_year":
        return bool(YEAR_RE.search(s))
    if field == "power_output":
        return bool(POWER_RE.search(s))
    if field == "serial_number":
        return bool(SERIAL_RE.search(s.upper()))
    if field == "model":
        return bool(MODEL_RE.search(s.upper()))
    if field == "brand":
        return len(s) >= 2 and any(c.isalpha() for c in s)
    return False


## 5. Heuristic field extraction

The same extractor is applied to the raw output of every OCR model. This isolates differences in OCR quality from differences in post processing. Patterns are written against general knowledge of typeplate layouts; no image from the testset was used to tune them.

In [ ]:
# Known brand vocabulary - used as a soft anchor for the brand field.
# Compiled from common typeplate manufacturers visible in the dataset.
KNOWN_BRANDS = {
    "westinghouse", "ge", "general electric", "siemens", "abb", "schneider",
    "mitsubishi", "toshiba", "hitachi", "ipg", "ipg laser",
    "baldor", "leeson", "leroy somer", "wege", "weg", "marathon",
    "nidec", "emerson", "honeywell", "yaskawa", "fanuc",
    "bosch", "atlas copco", "ingersoll rand", "danfoss", "grundfos",
    "ksb", "wilo", "lowara", "ebara", "viessmann", "vaillant",
    "remeha", "intergas", "nefit", "atag", "daikin", "mitsubishi electric",
    "carrier", "trane", "york", "lg", "samsung", "panasonic",
    "philips", "thomson", "lenze", "sew", "rexroth",
    "festo", "smc", "norgren", "parker", "eaton", "moeller",
}


def extract_brand(text):
    text_low = text.lower()
    # Direct vocabulary match
    for b in sorted(KNOWN_BRANDS, key=len, reverse=True):
        if b in text_low:
            return b.title()
    # Fallback: first all-caps word of length >= 3 on the first line
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue
        for tok in re.findall(r"\b[A-Z][A-Za-z\-]{2,}\b", line):
            return tok
        break
    return None


def extract_model(text):
    # Look for "model" label followed by a value
    m = re.search(r"(?:model|type|cat(?:alog)?\s*(?:no|number|num)?)[\s:.\-]*([A-Z0-9][A-Z0-9\-/\.]{2,})",
                  text, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()
    # Fallback: alphanumeric token with at least one digit and one letter
    for tok in re.findall(r"\b[A-Z0-9\-]{4,}\b", text.upper()):
        if any(c.isdigit() for c in tok) and any(c.isalpha() for c in tok):
            return tok
    return None


def extract_serial(text):
    m = re.search(r"(?:s\/?n|serial(?:\s*(?:no|number|num))?)[\s:.\-]*([A-Z0-9][A-Z0-9\-]{3,})",
                  text, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return None


def extract_year(text):
    # Years between 1950 and current
    matches = YEAR_RE.findall(text)
    if matches:
        # Prefer the most recent plausible year
        years = sorted({int(y) for y in matches})
        return str(years[-1])
    return None


def extract_power(text):
    """Prefer real power units (HP, kW, kVA, VA) over voltage/current (V, A, Hz)."""
    # First try power units
    power_units_strict = re.compile(
        r"\b\d+(?:[.,]\d+)?\s*(?:kw|kva|hp|va)\b", re.IGNORECASE)
    m = power_units_strict.search(text)
    if m:
        return m.group(0).strip()
    # Fall back to broader pattern (V, A, Hz, W)
    m = POWER_RE.search(text)
    if m:
        return m.group(0).strip()
    return None


def extract_all_fields(raw_text):
    if not raw_text:
        return {"brand": None, "model": None, "serial_number": None,
                "build_year": None, "power_output": None}
    return {
        "brand": extract_brand(raw_text),
        "model": extract_model(raw_text),
        "serial_number": extract_serial(raw_text),
        "build_year": extract_year(raw_text),
        "power_output": extract_power(raw_text),
    }


# Smoke test
sample = "WESTINGHOUSE Type AR-2 S/N 12345 Year 2003 240 V 1 HP"
print(extract_all_fields(sample))


## 6. OCR model runners

Each model is wrapped in a function that takes an image path and returns `(raw_text, latency_seconds)`. Errors are caught so a single failing image does not break the loop. Outputs are cached to disk.

In [ ]:
# ----- Tesseract -----------------------------------------------
def run_tesseract(image_path, psm=6, preprocessed=None):
    """Returns (raw_text, latency_seconds)."""
    t0 = time.time()
    try:
        if preprocessed is not None:
            img = preprocessed if isinstance(preprocessed, Image.Image) else Image.fromarray(preprocessed)
        else:
            img = Image.open(image_path).convert("L")
        text = pytesseract.image_to_string(img, config=f"--psm {psm}", lang="eng")
    except Exception as e:
        text = ""
        print(f"Tesseract error on {image_path}: {e}")
    return text, time.time() - t0


In [ ]:
# ----- EasyOCR (lazy init) -------------------------------------
_easyocr_reader = None

def get_easyocr_reader():
    global _easyocr_reader
    if _easyocr_reader is None:
        import torch
        gpu_ok = torch.cuda.is_available()
        try:
            _easyocr_reader = easyocr.Reader(["en"], gpu=gpu_ok, verbose=False)
            print(f"EasyOCR initialised (gpu={gpu_ok})")
        except Exception as e:
            print(f"EasyOCR GPU init failed: {e}; falling back to CPU")
            _easyocr_reader = easyocr.Reader(["en"], gpu=False, verbose=False)
    return _easyocr_reader


def run_easyocr(image_path, preprocessed=None):
    t0 = time.time()
    try:
        reader = get_easyocr_reader()
        if preprocessed is not None:
            arr = np.array(preprocessed) if isinstance(preprocessed, Image.Image) else preprocessed
        else:
            arr = np.array(Image.open(image_path).convert("RGB"))
        # detail=0 returns plain text; we want bbox info for ordering
        results = reader.readtext(arr, detail=1, paragraph=False)
        # Sort top to bottom, left to right
        results_sorted = sorted(results, key=lambda x: (x[0][0][1], x[0][0][0]))
        text = "\n".join(r[1] for r in results_sorted)
    except Exception as e:
        text = ""
        print(f"EasyOCR error on {image_path}: {e}")
    return text, time.time() - t0


In [ ]:
# ----- PaddleOCR (lazy init) -----------------------------------
_paddle_ocr = None

def get_paddle_ocr():
    global _paddle_ocr
    if _paddle_ocr is None:
        try:
            # use_angle_cls helps with rotated text
            _paddle_ocr = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
            print("PaddleOCR initialised")
        except TypeError:
            # Newer PaddleOCR API may not accept show_log
            _paddle_ocr = PaddleOCR(use_angle_cls=True, lang="en")
            print("PaddleOCR initialised (no show_log arg)")
    return _paddle_ocr


def run_paddleocr(image_path, preprocessed=None):
    t0 = time.time()
    try:
        ocr = get_paddle_ocr()
        if preprocessed is not None:
            arr = np.array(preprocessed) if isinstance(preprocessed, Image.Image) else preprocessed
            if arr.ndim == 2:
                arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
        else:
            arr = np.array(Image.open(image_path).convert("RGB"))
        # PaddleOCR API has changed across versions; handle both
        try:
            results = ocr.ocr(arr, cls=True)
        except TypeError:
            results = ocr.ocr(arr)
        # Flatten and sort
        lines = []
        if results and results[0]:
            for box_info in results[0]:
                if box_info and len(box_info) >= 2:
                    bbox, (txt, conf) = box_info[0], box_info[1]
                    y_top = min(p[1] for p in bbox)
                    x_left = min(p[0] for p in bbox)
                    lines.append((y_top, x_left, txt))
        lines.sort()
        text = "\n".join(t for _, _, t in lines)
    except Exception as e:
        text = ""
        print(f"PaddleOCR error on {image_path}: {e}")
    return text, time.time() - t0


In [ ]:
# ----- Cached batch runner -------------------------------------
def run_model_on_dataset(model_name, run_fn, df, image_dir, cache_tag,
                          preprocess_fn=None, force=False):
    """Run an OCR function on all images and cache the raw text."""
    cache_path = CACHE_DIR / f"raw_{model_name}_{cache_tag}.json"
    if cache_path.exists() and not force:
        with open(cache_path, "r", encoding="utf-8") as f:
            cached = json.load(f)
        print(f"Loaded cached results for {model_name} ({cache_tag}): {len(cached)} rows")
        return cached

    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{model_name} ({cache_tag})"):
        img_path = image_dir / row["filename"]
        if not img_path.exists():
            rows.append({"id": int(row["id"]), "filename": row["filename"],
                         "raw_text": "", "latency": 0.0, "ok": False,
                         "error": "image not found"})
            continue
        preprocessed = None
        if preprocess_fn is not None:
            try:
                preprocessed = preprocess_fn(str(img_path))
            except Exception as e:
                preprocessed = None
        try:
            text, lat = run_fn(str(img_path), preprocessed=preprocessed) if preprocess_fn else run_fn(str(img_path))
            rows.append({"id": int(row["id"]), "filename": row["filename"],
                         "raw_text": text, "latency": float(lat), "ok": True})
        except Exception as e:
            rows.append({"id": int(row["id"]), "filename": row["filename"],
                         "raw_text": "", "latency": 0.0, "ok": False, "error": str(e)})

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"Cached results to {cache_path}")
    return rows


In [ ]:
# ----- Evaluation routine --------------------------------------
def evaluate_run(raw_results, df, run_name):
    """Compute per row and aggregate metrics for one OCR run."""
    merged = []
    by_id = {r["id"]: r for r in raw_results}
    for _, row in df.iterrows():
        r = by_id.get(int(row["id"]))
        if r is None:
            continue
        raw = r.get("raw_text", "") or ""
        extracted = extract_all_fields(raw)
        merged.append({
            "id": int(row["id"]),
            "filename": row["filename"],
            "plate_type": row["plate_type"],
            "readability": row["readability"],
            "status": row["status"],
            "transcription": row["transcription"],
            "raw_text": raw,
            "latency": r.get("latency", 0.0),
            "gt_brand": to_str_or_none(row["brand"]), "pred_brand": extracted["brand"],
            "gt_model": to_str_or_none(row["model"]), "pred_model": extracted["model"],
            "gt_serial_number": to_str_or_none(row["serial_number"]), "pred_serial_number": extracted["serial_number"],
            "gt_build_year": to_str_or_none(row["build_year"]), "pred_build_year": extracted["build_year"],
            "gt_power_output": to_str_or_none(row["power_output"]), "pred_power_output": extracted["power_output"],
        })

    field_metrics = {}
    field_format_ok = {}
    for field in ["brand", "model", "serial_number", "build_year", "power_output"]:
        m = compute_field_metrics(merged, f"gt_{field}", f"pred_{field}")
        field_metrics[field] = m
        fmt_count = sum(1 for r in merged if format_valid(field, r[f"pred_{field}"]))
        gt_count = sum(1 for r in merged if not is_missing(r[f"gt_{field}"]))
        field_format_ok[field] = fmt_count / gt_count if gt_count > 0 else 0.0

    # Raw text CER/WER vs full transcription
    cer_vals = [safe_cer(r["transcription"], r["raw_text"]) for r in merged]
    wer_vals = [safe_wer(r["transcription"], r["raw_text"]) for r in merged]
    latencies = [r["latency"] for r in merged if r["latency"] > 0]

    summary = {
        "run": run_name,
        "n": len(merged),
        "cer_mean": float(np.mean(cer_vals)),
        "wer_mean": float(np.mean(wer_vals)),
        "latency_mean": float(np.mean(latencies)) if latencies else 0.0,
        "latency_median": float(np.median(latencies)) if latencies else 0.0,
        "macro_f1": float(np.mean([field_metrics[f]["f1"] for f in field_metrics])),
        "macro_fmtok": float(np.mean(list(field_format_ok.values()))),
    }
    for f, m in field_metrics.items():
        summary[f"{f}_precision"] = m["precision"]
        summary[f"{f}_recall"] = m["recall"]
        summary[f"{f}_f1"] = m["f1"]
        summary[f"{f}_tp"] = m["tp"]
        summary[f"{f}_fp"] = m["fp"]
        summary[f"{f}_fn"] = m["fn"]
        summary[f"{f}_fmtok"] = field_format_ok[f]

    return merged, summary


## 7. Experiment 1 — Baseline and candidate models

Run Tesseract (baseline), EasyOCR, and PaddleOCR on the raw test images without any preprocessing. Outputs are cached to disk so a rerun does not repeat the OCR step.

In [ ]:
# ----- Run Tesseract baseline ----------------------------------
tesseract_raw = run_model_on_dataset(
    "tesseract", run_tesseract, df, TESTSET_IMAGES_DIR, cache_tag="raw",
)
tesseract_rows, tesseract_summary = evaluate_run(tesseract_raw, df, "tesseract_raw")
print("Tesseract macro F1:", round(tesseract_summary["macro_f1"], 3))
print("Tesseract CER mean:", round(tesseract_summary["cer_mean"], 3))


In [ ]:
# ----- Run EasyOCR ---------------------------------------------
easyocr_raw = run_model_on_dataset(
    "easyocr", run_easyocr, df, TESTSET_IMAGES_DIR, cache_tag="raw",
)
easyocr_rows, easyocr_summary = evaluate_run(easyocr_raw, df, "easyocr_raw")
print("EasyOCR macro F1:", round(easyocr_summary["macro_f1"], 3))
print("EasyOCR CER mean:", round(easyocr_summary["cer_mean"], 3))


In [ ]:
# ----- Run PaddleOCR -------------------------------------------
paddleocr_raw = run_model_on_dataset(
    "paddleocr", run_paddleocr, df, TESTSET_IMAGES_DIR, cache_tag="raw",
)
paddleocr_rows, paddleocr_summary = evaluate_run(paddleocr_raw, df, "paddleocr_raw")
print("PaddleOCR macro F1:", round(paddleocr_summary["macro_f1"], 3))
print("PaddleOCR CER mean:", round(paddleocr_summary["cer_mean"], 3))


In [ ]:
# ----- Comparison table ----------------------------------------
summaries_exp1 = [tesseract_summary, easyocr_summary, paddleocr_summary]
exp1_df = pd.DataFrame(summaries_exp1)
exp1_df.to_csv(OUTPUT_DIR / "exp1_summary.csv", index=False)

# Pretty headline table
print("\nExperiment 1 headline numbers")
print("-" * 70)
print(f"{'Model':<22} {'macro F1':>10} {'CER':>10} {'WER':>10} {'lat (s)':>10}")
print("-" * 70)
for s in summaries_exp1:
    print(f"{s['run']:<22} {s['macro_f1']:>10.3f} {s['cer_mean']:>10.3f} "
          f"{s['wer_mean']:>10.3f} {s['latency_mean']:>10.3f}")
print("-" * 70)

# Per field F1 table
fields = ["brand", "model", "serial_number", "build_year", "power_output"]
per_field = pd.DataFrame({
    s["run"]: [s[f"{f}_f1"] for f in fields] for s in summaries_exp1
}, index=fields)
per_field.to_csv(OUTPUT_DIR / "exp1_per_field_f1.csv")
print("\nPer field F1")
print(per_field.round(3))

# Save per row details
for name, rows in [("tesseract", tesseract_rows), ("easyocr", easyocr_rows), ("paddleocr", paddleocr_rows)]:
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / f"exp1_rows_{name}.csv", index=False)


In [ ]:
# ----- Figure 7: CER and macro F1 comparison -------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

names = [s["run"].replace("_raw", "") for s in summaries_exp1]
cer_vals = [s["cer_mean"] for s in summaries_exp1]
f1_vals = [s["macro_f1"] for s in summaries_exp1]

axes[0].bar(names, cer_vals, color=["#d9534f", "#5cb85c", "#337ab7"])
axes[0].set_ylabel("Mean CER")
axes[0].set_title("Character Error Rate (lower is better)")
for i, v in enumerate(cer_vals):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center")

axes[1].bar(names, f1_vals, color=["#d9534f", "#5cb85c", "#337ab7"])
axes[1].axhline(0.75, ls="--", color="black", alpha=0.5, label="go/no-go = 0.75")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Field level macro F1 (higher is better)")
axes[1].set_ylim(0, 1)
axes[1].legend()
for i, v in enumerate(f1_vals):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig07_exp1_cer_f1.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ----- Figure 8: per field F1 heatmap --------------------------
fig, ax = plt.subplots(figsize=(8, 4))
data = per_field.values
im = ax.imshow(data, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(per_field.columns)))
ax.set_xticklabels(per_field.columns, rotation=20, ha="right")
ax.set_yticks(range(len(per_field.index)))
ax.set_yticklabels(per_field.index)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f"{data[i, j]:.2f}", ha="center", va="center",
                color="black" if data[i, j] > 0.3 else "white")
plt.colorbar(im, label="F1")
ax.set_title("Field level F1 per model")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig08_exp1_per_field_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ----- Figure 9: CER by plate type -----------------------------
def cer_by_platetype(rows):
    out = {}
    for pt in df["plate_type"].unique():
        sub = [r for r in rows if r["plate_type"] == pt]
        if not sub:
            continue
        out[pt] = float(np.mean([safe_cer(r["transcription"], r["raw_text"]) for r in sub]))
    return out

cer_breakdown = {
    "tesseract": cer_by_platetype(tesseract_rows),
    "easyocr": cer_by_platetype(easyocr_rows),
    "paddleocr": cer_by_platetype(paddleocr_rows),
}
breakdown_df = pd.DataFrame(cer_breakdown)
breakdown_df.to_csv(OUTPUT_DIR / "exp1_cer_by_platetype.csv")

fig, ax = plt.subplots(figsize=(9, 4.5))
breakdown_df.plot(kind="bar", ax=ax, color=["#d9534f", "#5cb85c", "#337ab7"])
ax.set_ylabel("Mean CER")
ax.set_title("CER by plate type and model")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig09_exp1_cer_by_platetype.png", dpi=150, bbox_inches="tight")
plt.show()
print(breakdown_df.round(3))


## 8. Preprocessing pipeline

Two operations are applied in sequence. Each is justified against the alternatives in the thesis (Section 3.4).

**Rotation correction (Hough line transform).** Detects the dominant straight lines via Canny + HoughLines and takes the median deviation from horizontal as the skew angle. Following Amin and Fischer (2000), Hough based detection is preferred over projection profile methods for non document images, where the text fills only a fraction of the frame and a strong background gradient destabilises the projection variance.

**Contrast enhancement (CLAHE).** Local histogram equalisation introduced by Zuiderveld (1994). The clip limit prevents the noise amplification of global histogram equalisation, which is a known failure mode on glossy or reflective surfaces. The license plate OCR study by Tavares (2024) reports that CLAHE does not uniformly help OCR accuracy across all plates, which matches hypothesis H4.

In [ ]:
def deskew_hough(gray):
    """Detect and correct image rotation using the Hough line transform.

    1. Canny edge map.
    2. HoughLines for straight lines.
    3. Median deviation from horizontal as the rotation angle.
    4. Rotate only if abs(angle) > 1 degree.
    """
    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=120)
    if lines is None:
        return gray, 0.0
    angles = []
    for line in lines[:50]:
        theta = line[0][1]
        deg = np.degrees(theta) - 90.0
        if -45 < deg < 45:
            angles.append(deg)
    if not angles:
        return gray, 0.0
    angle = float(np.median(angles))
    if abs(angle) < 1.0:
        return gray, angle
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h),
                             flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def apply_clahe(gray, clip_limit=2.0, tile_grid_size=(8, 8)):
    """CLAHE with the OpenCV defaults from the original Zuiderveld implementation."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(gray)


def preprocess_image_path(path):
    """Full pipeline: load -> grayscale -> Hough deskew -> CLAHE -> PIL."""
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        return Image.open(path).convert("L")
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    deskewed, angle = deskew_hough(gray)
    enhanced = apply_clahe(deskewed)
    return Image.fromarray(enhanced)


def preprocess_with_info(path):
    """Same pipeline, but also return the rotation angle for logging."""
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        return Image.open(path).convert("L"), 0.0
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    deskewed, angle = deskew_hough(gray)
    enhanced = apply_clahe(deskewed)
    return Image.fromarray(enhanced), angle


In [ ]:
# ----- Figure 6: example of preprocessing on one image ----------
EXAMPLE_FILE = None  # auto-pick first existing image
if EXAMPLE_FILE is None:
    candidates = sorted(TESTSET_IMAGES_DIR.glob("*.jpg")) + sorted(TESTSET_IMAGES_DIR.glob("*.png"))
    if candidates:
        EXAMPLE_FILE = candidates[0].name

if EXAMPLE_FILE:
    orig_path = TESTSET_IMAGES_DIR / EXAMPLE_FILE
    orig_pil = Image.open(orig_path).convert("L")
    prep_pil, angle = preprocess_with_info(orig_path)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(np.array(orig_pil), cmap="gray")
    axes[0].set_title(f"Original (greyscale)\n{EXAMPLE_FILE}", fontweight="bold")
    axes[0].axis("off")
    axes[1].imshow(np.array(prep_pil), cmap="gray")
    axes[1].set_title(f"After preprocessing\n(Hough deskew {angle:.1f}° + CLAHE)",
                      fontweight="bold")
    axes[1].axis("off")
    plt.suptitle("Figure 6: Effect of preprocessing on one example image",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig06_preprocessing_example.png",
                dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No images found in TESTSET_IMAGES_DIR.")


In [ ]:
# ----- Activation rate of rotation correction -------------------
rotation_log = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Rotation analysis"):
    p = TESTSET_IMAGES_DIR / row["filename"]
    if not p.exists():
        continue
    img = cv2.imread(str(p))
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, angle = deskew_hough(gray)
    rotation_log.append({"id": int(row["id"]), "filename": row["filename"],
                         "plate_type": row["plate_type"],
                         "rotation_angle": float(angle),
                         "rotation_applied": bool(abs(angle) >= 1.0)})

rot_df = pd.DataFrame(rotation_log)
rot_df.to_csv(OUTPUT_DIR / "exp2_rotation_log.csv", index=False)
print(f"Rotation correction triggered on {rot_df['rotation_applied'].sum()} / {len(rot_df)} images")
print("By plate type:")
print(rot_df.groupby("plate_type")["rotation_applied"].agg(["sum", "count"]))


## 9. Experiment 2 — Effect of preprocessing

Each of the three OCR models is rerun with the preprocessing pipeline applied to every image. Delta tables and figures isolate the contribution of preprocessing from the contribution of the model itself.

In [ ]:
# ----- Wrapper that injects preprocessed image into each runner -----
def run_tesseract_pre(image_path, preprocessed=None):
    if preprocessed is None:
        preprocessed = preprocess_image_path(image_path)
    return run_tesseract(image_path, preprocessed=preprocessed)


def run_easyocr_pre(image_path, preprocessed=None):
    if preprocessed is None:
        preprocessed = preprocess_image_path(image_path)
    return run_easyocr(image_path, preprocessed=preprocessed)


def run_paddleocr_pre(image_path, preprocessed=None):
    if preprocessed is None:
        preprocessed = preprocess_image_path(image_path)
    return run_paddleocr(image_path, preprocessed=preprocessed)


In [ ]:
# ----- Tesseract with preprocessing ----------------------------
tesseract_pre_raw = run_model_on_dataset(
    "tesseract", run_tesseract_pre, df, TESTSET_IMAGES_DIR, cache_tag="pre",
)
tesseract_pre_rows, tesseract_pre_summary = evaluate_run(
    tesseract_pre_raw, df, "tesseract_pre"
)
print("Tesseract (pre) macro F1:", round(tesseract_pre_summary["macro_f1"], 3))


In [ ]:
# ----- EasyOCR with preprocessing ------------------------------
easyocr_pre_raw = run_model_on_dataset(
    "easyocr", run_easyocr_pre, df, TESTSET_IMAGES_DIR, cache_tag="pre",
)
easyocr_pre_rows, easyocr_pre_summary = evaluate_run(easyocr_pre_raw, df, "easyocr_pre")
print("EasyOCR (pre) macro F1:", round(easyocr_pre_summary["macro_f1"], 3))


In [ ]:
# ----- PaddleOCR with preprocessing ----------------------------
paddleocr_pre_raw = run_model_on_dataset(
    "paddleocr", run_paddleocr_pre, df, TESTSET_IMAGES_DIR, cache_tag="pre",
)
paddleocr_pre_rows, paddleocr_pre_summary = evaluate_run(paddleocr_pre_raw, df, "paddleocr_pre")
print("PaddleOCR (pre) macro F1:", round(paddleocr_pre_summary["macro_f1"], 3))


In [ ]:
# ----- Delta table ---------------------------------------------
exp2_summaries = [
    tesseract_summary, tesseract_pre_summary,
    easyocr_summary, easyocr_pre_summary,
    paddleocr_summary, paddleocr_pre_summary,
]
exp2_df = pd.DataFrame(exp2_summaries)
exp2_df.to_csv(OUTPUT_DIR / "exp2_summary.csv", index=False)

print("\nExperiment 2 - with and without preprocessing")
print("-" * 70)
print(f"{'Run':<22} {'macro F1':>10} {'CER':>10} {'lat (s)':>10}")
print("-" * 70)
for s in exp2_summaries:
    print(f"{s['run']:<22} {s['macro_f1']:>10.3f} {s['cer_mean']:>10.3f} {s['latency_mean']:>10.3f}")
print("-" * 70)

# Compute deltas (preprocessed minus raw)
deltas = []
for model_name, raw_s, pre_s in [
    ("tesseract", tesseract_summary, tesseract_pre_summary),
    ("easyocr", easyocr_summary, easyocr_pre_summary),
    ("paddleocr", paddleocr_summary, paddleocr_pre_summary),
]:
    deltas.append({
        "model": model_name,
        "delta_macro_f1": pre_s["macro_f1"] - raw_s["macro_f1"],
        "delta_cer": pre_s["cer_mean"] - raw_s["cer_mean"],
        "delta_wer": pre_s["wer_mean"] - raw_s["wer_mean"],
        "delta_latency": pre_s["latency_mean"] - raw_s["latency_mean"],
    })

delta_df = pd.DataFrame(deltas)
delta_df.to_csv(OUTPUT_DIR / "exp2_deltas.csv", index=False)
print("\nDeltas (preprocessed minus raw)")
print(delta_df.round(3))


In [ ]:
# ----- Figure 10: with-vs-without preprocessing ----------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

models = ["tesseract", "easyocr", "paddleocr"]
raw_f1 = [tesseract_summary["macro_f1"], easyocr_summary["macro_f1"], paddleocr_summary["macro_f1"]]
pre_f1 = [tesseract_pre_summary["macro_f1"], easyocr_pre_summary["macro_f1"], paddleocr_pre_summary["macro_f1"]]

x = np.arange(len(models))
width = 0.35
axes[0].bar(x - width/2, raw_f1, width, label="No preprocessing", color="#d9534f")
axes[0].bar(x + width/2, pre_f1, width, label="With preprocessing", color="#5cb85c")
axes[0].axhline(0.75, ls="--", color="black", alpha=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].set_ylabel("Macro F1")
axes[0].set_title("Macro F1: with vs without preprocessing")
axes[0].legend()
axes[0].set_ylim(0, 1)

raw_cer = [tesseract_summary["cer_mean"], easyocr_summary["cer_mean"], paddleocr_summary["cer_mean"]]
pre_cer = [tesseract_pre_summary["cer_mean"], easyocr_pre_summary["cer_mean"], paddleocr_pre_summary["cer_mean"]]
axes[1].bar(x - width/2, raw_cer, width, label="No preprocessing", color="#d9534f")
axes[1].bar(x + width/2, pre_cer, width, label="With preprocessing", color="#5cb85c")
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].set_ylabel("Mean CER")
axes[1].set_title("CER: with vs without preprocessing")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig10_exp2_preprocessing_impact.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ----- Per plate type delta ------------------------------------
def f1_by_platetype(rows):
    out = {}
    for pt in df["plate_type"].unique():
        sub = [r for r in rows if r["plate_type"] == pt]
        if not sub:
            continue
        # compute macro F1 over the subset
        local_f1 = []
        for field in ["brand", "model", "serial_number", "build_year", "power_output"]:
            m = compute_field_metrics(sub, f"gt_{field}", f"pred_{field}")
            local_f1.append(m["f1"])
        out[pt] = float(np.mean(local_f1))
    return out

plate_breakdown = pd.DataFrame({
    "tesseract_raw": f1_by_platetype(tesseract_rows),
    "tesseract_pre": f1_by_platetype(tesseract_pre_rows),
    "easyocr_raw": f1_by_platetype(easyocr_rows),
    "easyocr_pre": f1_by_platetype(easyocr_pre_rows),
    "paddleocr_raw": f1_by_platetype(paddleocr_rows),
    "paddleocr_pre": f1_by_platetype(paddleocr_pre_rows),
})
plate_breakdown.to_csv(OUTPUT_DIR / "exp2_f1_by_platetype.csv")
print("Macro F1 by plate type (raw vs preprocessed)")
print(plate_breakdown.round(3))


## 10. Experiment 3 — Offline vs online comparison (optional)

This experiment places the best offline configuration from Experiments 1 and 2 against an online vision language model. Running it requires an API key. If you do not have one, this section can be skipped — the literature based numbers in `exp3_literature_reference.csv` are sufficient for the comparison narrative.

To enable: set `RUN_EXP3 = True` and provide an API key for either Anthropic (Claude) or OpenAI (GPT-4o).

A small sample of 20 images is used to keep API costs around 1 USD.

In [ ]:
# ----- Configuration -------------------------------------------
RUN_EXP3 = False  # set to True to run the online comparison
EXP3_PROVIDER = "anthropic"  # "anthropic" or "openai"
EXP3_SAMPLE_SIZE = 20

# Keys: leave empty here and use environment variables ANTHROPIC_API_KEY or OPENAI_API_KEY
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")


In [ ]:
# ----- Online VLM call (Claude or GPT-4o) ----------------------
import base64

def encode_image_b64(path):
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")


VLM_PROMPT = """You are reading a typeplate from an industrial installation.
Extract the following fields and return them as JSON only (no other text):

{
  "brand": "<manufacturer name or null>",
  "model": "<model code or null>",
  "serial_number": "<serial number or null>",
  "build_year": "<four digit year or null>",
  "power_output": "<value with unit, e.g. '45.0 kVA' or null>"
}

Use null when a field is not visible on the plate."""


def run_claude(image_path):
    """Call Anthropic Claude with the image. Requires ANTHROPIC_API_KEY."""
    try:
        import anthropic
    except ImportError:
        raise RuntimeError("pip install anthropic")
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    img_b64 = encode_image_b64(image_path)
    ext = Path(image_path).suffix.lower().replace(".", "")
    media_type = "image/jpeg" if ext in {"jpg", "jpeg"} else "image/png"
    t0 = time.time()
    msg = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {
                    "type": "base64", "media_type": media_type, "data": img_b64
                }},
                {"type": "text", "text": VLM_PROMPT}
            ]
        }]
    )
    lat = time.time() - t0
    return msg.content[0].text, lat


def run_openai(image_path):
    """Call OpenAI GPT-4o with the image. Requires OPENAI_API_KEY."""
    try:
        from openai import OpenAI
    except ImportError:
        raise RuntimeError("pip install openai")
    client = OpenAI(api_key=OPENAI_API_KEY)
    img_b64 = encode_image_b64(image_path)
    ext = Path(image_path).suffix.lower().replace(".", "")
    mime = "image/jpeg" if ext in {"jpg", "jpeg"} else "image/png"
    t0 = time.time()
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": VLM_PROMPT},
                {"type": "image_url",
                 "image_url": {"url": f"data:{mime};base64,{img_b64}"}}
            ]
        }]
    )
    lat = time.time() - t0
    return resp.choices[0].message.content, lat


def parse_vlm_json(text):
    """Extract JSON dict from a VLM response."""
    text = text.strip()
    # Strip code fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except Exception:
        # Try to find a JSON object inside the text
        m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return {}
        return {}


In [ ]:
# ----- Run Experiment 3 (if enabled) ---------------------------
if RUN_EXP3:
    if EXP3_PROVIDER == "anthropic" and not ANTHROPIC_API_KEY:
        print("Set ANTHROPIC_API_KEY in your environment to run.")
    elif EXP3_PROVIDER == "openai" and not OPENAI_API_KEY:
        print("Set OPENAI_API_KEY in your environment to run.")
    else:
        # Sample stratified by plate_type if possible
        sample_df = df.sample(n=min(EXP3_SAMPLE_SIZE, len(df)),
                              random_state=42).reset_index(drop=True)
        vlm_rows = []
        runner = run_claude if EXP3_PROVIDER == "anthropic" else run_openai
        for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="VLM"):
            p = TESTSET_IMAGES_DIR / row["filename"]
            if not p.exists():
                continue
            try:
                txt, lat = runner(str(p))
                parsed = parse_vlm_json(txt)
            except Exception as e:
                txt, lat, parsed = str(e), 0.0, {}
            vlm_rows.append({
                "id": int(row["id"]),
                "filename": row["filename"],
                "plate_type": row["plate_type"],
                "raw_response": txt,
                "latency": lat,
                "transcription": row["transcription"],
                "gt_brand": to_str_or_none(row["brand"]), "pred_brand": parsed.get("brand"),
                "gt_model": to_str_or_none(row["model"]), "pred_model": parsed.get("model"),
                "gt_serial_number": to_str_or_none(row["serial_number"]), "pred_serial_number": parsed.get("serial_number"),
                "gt_build_year": to_str_or_none(row["build_year"]), "pred_build_year": parsed.get("build_year"),
                "gt_power_output": to_str_or_none(row["power_output"]), "pred_power_output": parsed.get("power_output"),
            })

        vlm_df = pd.DataFrame(vlm_rows)
        vlm_df.to_csv(OUTPUT_DIR / "exp3_vlm_rows.csv", index=False)

        # Field metrics
        field_metrics_vlm = {}
        for field in ["brand", "model", "serial_number", "build_year", "power_output"]:
            field_metrics_vlm[field] = compute_field_metrics(vlm_rows, f"gt_{field}", f"pred_{field}")
        vlm_macro_f1 = float(np.mean([m["f1"] for m in field_metrics_vlm.values()]))
        print(f"VLM ({EXP3_PROVIDER}) macro F1 on {len(vlm_rows)} sample: {vlm_macro_f1:.3f}")
        print(f"VLM mean latency: {np.mean([r['latency'] for r in vlm_rows]):.2f} s")

        # Save summary
        vlm_summary = {
            "provider": EXP3_PROVIDER,
            "n_sample": len(vlm_rows),
            "macro_f1": vlm_macro_f1,
            "latency_mean": float(np.mean([r["latency"] for r in vlm_rows])),
        }
        for f, m in field_metrics_vlm.items():
            vlm_summary[f"{f}_f1"] = m["f1"]
        pd.DataFrame([vlm_summary]).to_csv(OUTPUT_DIR / "exp3_vlm_summary.csv", index=False)
else:
    print("Experiment 3 skipped. To enable, set RUN_EXP3 = True at the top of this section.")


In [ ]:
# ----- Comparison table (offline best vs online VLM) -----------
# Best offline configuration is automatically picked as the one with highest macro F1
all_runs = [
    ("tesseract_raw", tesseract_summary),
    ("tesseract_pre", tesseract_pre_summary),
    ("easyocr_raw", easyocr_summary),
    ("easyocr_pre", easyocr_pre_summary),
    ("paddleocr_raw", paddleocr_summary),
    ("paddleocr_pre", paddleocr_pre_summary),
]
best_name, best_summary = max(all_runs, key=lambda x: x[1]["macro_f1"])
print(f"Best offline configuration: {best_name} (macro F1 = {best_summary['macro_f1']:.3f})")

# Build comparison table including literature numbers for the offline/online tradeoff narrative
comparison = pd.DataFrame([{
    "config": best_name,
    "macro_f1": best_summary["macro_f1"],
    "cer_mean": best_summary["cer_mean"],
    "latency_mean_s": best_summary["latency_mean"],
    "connectivity_required": "no",
    "monetary_cost_per_image": "0",
    "notes": "best offline configuration on testset"
}])

# Append VLM line if available
vlm_csv = OUTPUT_DIR / "exp3_vlm_summary.csv"
if vlm_csv.exists():
    vlm_summary_df = pd.read_csv(vlm_csv)
    if len(vlm_summary_df) > 0:
        v = vlm_summary_df.iloc[0]
        comparison = pd.concat([comparison, pd.DataFrame([{
            "config": f"online_{v['provider']}",
            "macro_f1": v["macro_f1"],
            "cer_mean": None,
            "latency_mean_s": v["latency_mean"],
            "connectivity_required": "yes",
            "monetary_cost_per_image": "approx 0.001-0.01 USD",
            "notes": f"measured on {int(v['n_sample'])} image sample"
        }])], ignore_index=True)

comparison.to_csv(OUTPUT_DIR / "exp3_comparison.csv", index=False)
print("\nOffline vs online comparison")
print(comparison.to_string(index=False))


## 11. Final summary and exports

All CSV and PNG outputs are written to `results/`. Hand these to the thesis writing step.

In [ ]:
print("Files in", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name, f"({p.stat().st_size:,} bytes)")

# Master summary file
master = pd.concat([
    pd.DataFrame(exp2_summaries),
], ignore_index=True)
master.to_csv(OUTPUT_DIR / "master_summary.csv", index=False)
print("\nAll done. Hand the results/ folder back to the thesis writing step.")
